# Hedge Fund OS — Backtest Exploration

Interactive notebook for running, analyzing, and iterating on the breakout strategy.

**What this notebook does:**
1. Downloads market data (SPY + stock universe)
2. Computes all technical features
3. Runs the backtest engine
4. Visualizes equity curve, drawdown, trade distribution
5. Lets you tweak parameters and re-run instantly

In [ ]:
# Setup — run this cell first
import sys
import os

# Add project root to path so imports work
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')  # Change to 'default' if you prefer light
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Setup complete.')

## 1. Download Market Data

In [ ]:
from src.data.ingest import download_ohlcv, get_spy, DEFAULT_UNIVERSE
from src.data.features import compute_all_features, compute_spy_features

# === CONFIGURE YOUR UNIVERSE HERE ===
START_DATE = '2015-01-01'
END_DATE = '2025-12-31'
TICKERS = DEFAULT_UNIVERSE[:50]  # Start with 50, increase later

print(f'Downloading SPY + {len(TICKERS)} stocks from {START_DATE} to {END_DATE}...')

# Download SPY
spy_raw = get_spy(start=START_DATE, end=END_DATE)
print(f'SPY: {len(spy_raw)} days')

# Download stocks
stock_data_raw = {}
for i, ticker in enumerate(TICKERS):
    try:
        df = download_ohlcv(ticker, start=START_DATE, end=END_DATE)
        if not df.empty and len(df) > 252:
            stock_data_raw[ticker] = df
    except Exception as e:
        pass
    if (i + 1) % 20 == 0:
        print(f'  Downloaded {i+1}/{len(TICKERS)}...')

print(f'\nGot {len(stock_data_raw)}/{len(TICKERS)} tickers')

## 2. Compute Features

In [ ]:
# Compute features for SPY and all stocks
spy = compute_spy_features(spy_raw)

stock_data = {}
for ticker, df in stock_data_raw.items():
    try:
        stock_data[ticker] = compute_all_features(df)
    except Exception as e:
        print(f'  Failed: {ticker} — {e}')

print(f'Features computed for {len(stock_data)} tickers')

# Quick look at one stock's features
sample_ticker = list(stock_data.keys())[0]
print(f'\nSample features for {sample_ticker}:')
stock_data[sample_ticker][['close', 'ma_150', 'ma_200', 'atr', 'vol_ratio', 'trend_valid', 'in_base', 'breakout']].tail(10)

## 3. Run the Backtest

**Tweak the parameters below and re-run this cell to iterate.**

In [ ]:
from src.backtest.engine import BacktestEngine, BacktestConfig

# ============================================================
# STRATEGY PARAMETERS — CHANGE THESE AND RE-RUN
# ============================================================
config = BacktestConfig(
    # Capital
    initial_capital=100_000,
    risk_per_trade=0.01,        # 1% risk per trade
    max_positions=10,
    max_position_pct=0.20,
    
    # Trend filter
    ma_fast=150,
    ma_slow=200,
    
    # Base detection
    base_min_len=15,
    base_max_len=40,
    base_max_depth=0.10,        # Try: 0.08, 0.10, 0.12
    
    # Breakout
    breakout_lookback=20,
    min_volume_ratio=1.5,       # Try: 1.5, 2.0, 2.5
    min_close_range_pct=0.25,
    
    # Stop loss
    stop_atr_mult=0.5,
    max_stop_pct=0.08,
    
    # Exits
    failed_breakout_days=5,     # Try: 3, 5, 7
    trailing_stop_lookback=10,  # Try: 7, 10, 15
    max_hold_days=60,
    
    # Execution realism
    slippage_pct=0.0005,
    commission_per_share=0.005,
)

# Run backtest
engine = BacktestEngine(config)
metrics = engine.run(stock_data, spy)
engine.print_summary(metrics)

# Get results as DataFrames
equity_curve = engine.get_equity_curve()
trade_log = engine.get_trade_log()

print(f'\nTrade log: {len(trade_log)} trades')

## 4. Equity Curve & Drawdown

In [ ]:
if not equity_curve.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle('Breakout Strategy — Equity Curve & Drawdown', fontsize=16, fontweight='bold')
    
    # Equity curve
    ax1.plot(equity_curve['date'], equity_curve['equity'], color='#3B82F6', linewidth=1.5, label='Strategy')
    ax1.axhline(y=config.initial_capital, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
    ax1.set_ylabel('Equity ($)')
    ax1.legend(loc='upper left')
    ax1.grid(alpha=0.2)
    ax1.xaxis.set_major_locator(mdates.YearLocator())
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    
    # Drawdown
    ax2.fill_between(equity_curve['date'], equity_curve['drawdown'] * 100, 0, 
                     color='#EF4444', alpha=0.5)
    ax2.set_ylabel('Drawdown (%)')
    ax2.set_xlabel('Date')
    ax2.grid(alpha=0.2)
    ax2.xaxis.set_major_locator(mdates.YearLocator())
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    
    plt.tight_layout()
    plt.show()
else:
    print('No equity curve data. Check if any trades were generated.')

## 5. Trade Analysis

In [ ]:
if not trade_log.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Trade Analysis', fontsize=16, fontweight='bold')
    
    # 1. PnL Distribution
    ax = axes[0, 0]
    colors = ['#22C55E' if x > 0 else '#EF4444' for x in trade_log['pnl_pct']]
    ax.bar(range(len(trade_log)), trade_log['pnl_pct'], color=colors, alpha=0.7, width=1.0)
    ax.set_title('PnL per trade (%)')
    ax.set_xlabel('Trade #')
    ax.axhline(y=0, color='white', linewidth=0.5)
    ax.grid(alpha=0.2)
    
    # 2. Exit Reason Breakdown
    ax = axes[0, 1]
    exit_counts = trade_log['exit_reason'].value_counts()
    colors_pie = ['#3B82F6', '#EF4444', '#F59E0B', '#22C55E', '#8B5CF6', '#EC4899']
    ax.pie(exit_counts.values, labels=exit_counts.index, autopct='%1.0f%%',
           colors=colors_pie[:len(exit_counts)], textprops={'fontsize': 10})
    ax.set_title('Exit reasons')
    
    # 3. Hold Time Distribution
    ax = axes[1, 0]
    ax.hist(trade_log['hold_days'], bins=20, color='#3B82F6', alpha=0.7, edgecolor='white')
    ax.set_title('Hold time distribution')
    ax.set_xlabel('Days held')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.2)
    
    # 4. Winners vs Losers
    ax = axes[1, 1]
    winners = trade_log[trade_log['pnl_pct'] > 0]
    losers = trade_log[trade_log['pnl_pct'] <= 0]
    stats = {
        'Count': [len(winners), len(losers)],
        'Avg %': [winners['pnl_pct'].mean() if len(winners) > 0 else 0,
                  losers['pnl_pct'].mean() if len(losers) > 0 else 0],
    }
    x = np.arange(2)
    ax.bar(x - 0.15, stats['Count'], 0.3, label='Count', color='#3B82F6', alpha=0.8)
    ax2 = ax.twinx()
    ax2.bar(x + 0.15, stats['Avg %'], 0.3, label='Avg PnL %', color='#F59E0B', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(['Winners', 'Losers'])
    ax.set_ylabel('Count')
    ax2.set_ylabel('Avg PnL %')
    ax.set_title('Winners vs losers')
    ax.grid(alpha=0.2)
    
    plt.tight_layout()
    plt.show()
    
    # Print top winners and losers
    print('\n=== TOP 5 WINNERS ===')
    print(trade_log.nlargest(5, 'pnl_pct')[['ticker', 'entry_date', 'exit_date', 'pnl_pct', 'hold_days', 'exit_reason']].to_string(index=False))
    
    print('\n=== TOP 5 LOSERS ===')
    print(trade_log.nsmallest(5, 'pnl_pct')[['ticker', 'entry_date', 'exit_date', 'pnl_pct', 'hold_days', 'exit_reason']].to_string(index=False))
else:
    print('No trades to analyze.')

## 6. Monthly Returns Heatmap

In [ ]:
if not equity_curve.empty:
    eq = equity_curve.copy()
    eq['date'] = pd.to_datetime(eq['date'])
    eq = eq.set_index('date')
    
    # Monthly returns
    monthly = eq['equity'].resample('M').last().pct_change() * 100
    monthly_df = pd.DataFrame({
        'year': monthly.index.year,
        'month': monthly.index.month,
        'return': monthly.values
    }).dropna()
    
    pivot = monthly_df.pivot_table(index='year', columns='month', values='return')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-5, vmax=5)
    
    ax.set_xticks(range(12))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    
    # Add text annotations
    for i in range(len(pivot.index)):
        for j in range(12):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=9,
                       color='black' if abs(val) < 3 else 'white')
    
    plt.colorbar(im, label='Monthly Return (%)')
    ax.set_title('Monthly Returns Heatmap (%)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 7. Parameter Sensitivity Test

Test multiple parameter combinations to find what matters most.

In [ ]:
# Volume ratio sensitivity test
print('Testing volume ratio sensitivity...')
print(f'{"Vol Ratio":>12s} {"Trades":>8s} {"Return":>10s} {"Sharpe":>8s} {"MaxDD":>10s} {"WinRate":>10s} {"PF":>8s}')
print('-' * 72)

for vol_ratio in [1.0, 1.25, 1.5, 2.0, 2.5, 3.0]:
    cfg = BacktestConfig(
        initial_capital=100_000,
        risk_per_trade=0.01,
        max_positions=10,
        min_volume_ratio=vol_ratio,
        base_max_depth=0.10,
        failed_breakout_days=5,
    )
    eng = BacktestEngine(cfg)
    m = eng.run(stock_data, spy)
    print(f'{vol_ratio:12.1f} {m.total_trades:8d} {m.total_return_pct:+9.2f}% {m.sharpe_ratio:8.2f} {m.max_drawdown_pct:9.2f}% {m.win_rate:9.1f}% {m.profit_factor:8.2f}')

In [ ]:
# Failed breakout days sensitivity test
print('Testing failed breakout exit window...')
print(f'{"FB Days":>12s} {"Trades":>8s} {"Return":>10s} {"Sharpe":>8s} {"MaxDD":>10s} {"WinRate":>10s} {"PF":>8s}')
print('-' * 72)

for fb_days in [2, 3, 5, 7, 10]:
    cfg = BacktestConfig(
        initial_capital=100_000,
        risk_per_trade=0.01,
        max_positions=10,
        min_volume_ratio=1.5,
        base_max_depth=0.10,
        failed_breakout_days=fb_days,
    )
    eng = BacktestEngine(cfg)
    m = eng.run(stock_data, spy)
    print(f'{fb_days:12d} {m.total_trades:8d} {m.total_return_pct:+9.2f}% {m.sharpe_ratio:8.2f} {m.max_drawdown_pct:9.2f}% {m.win_rate:9.1f}% {m.profit_factor:8.2f}')

In [ ]:
# Base depth sensitivity test
print('Testing base depth threshold...')
print(f'{"Base Depth":>12s} {"Trades":>8s} {"Return":>10s} {"Sharpe":>8s} {"MaxDD":>10s} {"WinRate":>10s} {"PF":>8s}')
print('-' * 72)

for depth in [0.06, 0.08, 0.10, 0.12, 0.15]:
    cfg = BacktestConfig(
        initial_capital=100_000,
        risk_per_trade=0.01,
        max_positions=10,
        min_volume_ratio=1.5,
        base_max_depth=depth,
        failed_breakout_days=5,
    )
    eng = BacktestEngine(cfg)
    m = eng.run(stock_data, spy)
    print(f'{depth:11.0%} {m.total_trades:8d} {m.total_return_pct:+9.2f}% {m.sharpe_ratio:8.2f} {m.max_drawdown_pct:9.2f}% {m.win_rate:9.1f}% {m.profit_factor:8.2f}')

## 8. Examine Individual Trades

Look at specific trades to understand what's happening.

In [ ]:
if not trade_log.empty:
    # Show full trade log sorted by date
    print(f'Total trades: {len(trade_log)}')
    print(f'\nMost recent 20 trades:')
    display_cols = ['ticker', 'entry_date', 'exit_date', 'entry_price', 'exit_price',
                    'pnl_pct', 'hold_days', 'exit_reason']
    print(trade_log[display_cols].tail(20).to_string(index=False))
    
    # Ticker frequency
    print('\n=== MOST TRADED TICKERS ===')
    ticker_counts = trade_log['ticker'].value_counts().head(10)
    for ticker, count in ticker_counts.items():
        ticker_trades = trade_log[trade_log['ticker'] == ticker]
        avg_pnl = ticker_trades['pnl_pct'].mean()
        print(f'  {ticker:6s}  trades: {count:3d}  avg pnl: {avg_pnl:+.2f}%')

## 9. Save Results

In [ ]:
# Save results to reports folder
import os
reports_dir = os.path.join(project_root, 'reports', 'backtests')
os.makedirs(reports_dir, exist_ok=True)

if not equity_curve.empty:
    equity_curve.to_csv(os.path.join(reports_dir, 'equity_curve.csv'), index=False)
    print('Saved equity_curve.csv')

if not trade_log.empty:
    trade_log.to_csv(os.path.join(reports_dir, 'trade_log.csv'), index=False)
    print('Saved trade_log.csv')

# Save metrics
with open(os.path.join(reports_dir, 'metrics.txt'), 'w') as f:
    for k, v in metrics.__dict__.items():
        f.write(f'{k}: {v}\n')
    print('Saved metrics.txt')

print('\nDone. Results saved to reports/backtests/')